In [ ]:
# ============================================================================
# PHẦN 1: CÀI ĐÁT THƯ VIỆN VÀ IMPORT
# ============================================================================

import random

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import ImageDraw
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from copy import deepcopy
import time
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, accuracy_score
import seaborn as sns

# Kiểm tra GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Đang sử dụng device: {device}")

# Thiết lập seed cho reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [ ]:
# ============================================================================
# PHẦN 2: ĐỊNH NGHĨA MODEL RESNET18 CHO CIFAR-10
# ============================================================================

def get_resnet18(num_classes=10, pretrained=False):
    """
    ResNet18 cho CIFAR-10 (32x32 images, 3 channels)
    Điều chỉnh layer đầu để phù hợp với kích thước ảnh nhỏ
    """
    if pretrained:
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = resnet18(weights=None)
    
    # Điều chỉnh conv1 cho CIFAR-10 (32x32) thay vì ImageNet (224x224)
    # kernel_size=3, stride=1, padding=1 (giữ kích thước)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    
    # Bỏ max pooling đầu tiên (không cần cho ảnh nhỏ)
    model.maxpool = nn.Identity()
    
    # Thay đổi fully connected layer cuối để output 10 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

# Tạo wrapper để tương thích với code cũ (return_features)
class ResNet18Wrapper(nn.Module):
    def __init__(self, num_classes=10, pretrained=False):
        super(ResNet18Wrapper, self).__init__()
        self.model = get_resnet18(num_classes, pretrained)
    
    def forward(self, x, return_features=False):
        if return_features:
            # Extract features before final FC layer
            x = self.model.conv1(x)
            x = self.model.bn1(x)
            x = self.model.relu(x)
            x = self.model.maxpool(x)
            
            x = self.model.layer1(x)
            x = self.model.layer2(x)
            x = self.model.layer3(x)
            x = self.model.layer4(x)
            
            x = self.model.avgpool(x)
            features = torch.flatten(x, 1)
            output = self.model.fc(features)
            return output, features
        else:
            return self.model(x)

In [ ]:
# ============================================================================
# PHẦN 3: LOAD VÀ CHUẨN BỊ DỮ LIỆU CIFAR-10 (tối ưu DataLoader + giữ raw cho backdoor)
# ============================================================================

print("Đang tải CIFAR-10 dataset...")

# Tham số dùng chung
BATCH_SIZE = 128
TEST_BATCH = 128
SEED = 42
NUM_WORKERS = 2                         # tăng tốc I/O
PIN_MEMORY = torch.cuda.is_available()  # copy nhanh hơn về GPU
PERSIST = True if NUM_WORKERS > 0 else False

# CIFAR-10 normalization (mean và std cho từng channel RGB)
# Tính từ training set của CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Datasets (normalized cho train/test)
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# Raw datasets (PIL) để đóng dấu backdoor khi cần
train_dataset_raw = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=None
)
test_dataset_raw = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=None
)

# Split 50k -> 45k / 5k (validation)
from torch.utils.data import random_split
generator = torch.Generator().manual_seed(SEED)
train_clean_dataset, val_dataset = random_split(train_dataset, [45000, 5000], generator=generator)

# DataLoaders (tối ưu I/O)
train_loader = DataLoader(
    train_clean_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSIST
)
val_loader = DataLoader(
    val_dataset, batch_size=TEST_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSIST
)
test_loader = DataLoader(
    test_dataset, batch_size=TEST_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSIST
)

# Danh sách (tensor, label) để tiện tạo poison label-flip (đã normalized)
train_items = [train_clean_dataset[i] for i in range(len(train_clean_dataset))]

# Raw items (PIL, label) tương ứng 45k để đóng dấu backdoor đúng vị trí
if hasattr(train_clean_dataset, "indices"):  # random_split giữ .indices
    train_clean_indices = train_clean_dataset.indices
else:
    train_clean_indices = list(range(45000))

train_raw_items = [train_dataset_raw[i] for i in train_clean_indices]

print(f"Train(clean) samples: {len(train_clean_dataset)} (45k)")
print(f"Validation samples: {len(val_dataset)} (5k)")
print(f"Test samples: {len(test_dataset)} (10k)")

# CIFAR-10 class names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']

# ============================================================================
# VISUALIZATION: DATASET TABLES (Matplotlib)
# ============================================================================

def get_label_counts(dataset, class_names):
    counts = {name: 0 for name in class_names}
    # Handle Subset vs Dataset
    if isinstance(dataset, torch.utils.data.Subset):
        targets = [dataset.dataset.targets[i] for i in dataset.indices]
    elif hasattr(dataset, 'targets'):
        targets = dataset.targets
    else:
        # Fallback (slow)
        targets = [y for _, y in dataset]
        
    for label in targets:
        counts[class_names[label]] += 1
    return counts

# 1. Calculate Distributions
train_counts = get_label_counts(train_clean_dataset, cifar10_classes)
val_counts = get_label_counts(val_dataset, cifar10_classes)
test_counts = get_label_counts(test_dataset, cifar10_classes)

# Create DataFrame
df_dist = pd.DataFrame({
    'Class': cifar10_classes,
    'Train (45k)': [train_counts[c] for c in cifar10_classes],
    'Val (5k)': [val_counts[c] for c in cifar10_classes],
    'Test (10k)': [test_counts[c] for c in cifar10_classes]
})
# Add Total row
df_dist.loc['Total'] = df_dist.sum(numeric_only=True)
df_dist.at['Total', 'Class'] = 'TOTAL'

# 2. Plot Label Distribution Table
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('tight')
ax.axis('off')
ax.set_title("Dataset Label Distribution", fontsize=14, fontweight='bold', y=1.02)

table = ax.table(cellText=df_dist.values,
                 colLabels=df_dist.columns,
                 cellLoc='center',
                 loc='center',
                 colColours=['#f2f2f2']*len(df_dist.columns))

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# Style
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='w')
    elif row == len(df_dist): # Total row (index is len because header is 0)
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#e6e6e6')

plt.show()

# 3. Plot Dataset Description Table
dataset_info = [
    ["Dataset Name", "CIFAR-10"],
    ["Image Size", "32x32 pixels"],
    ["Channels", "3 (RGB)"],
    ["Classes", "10"],
    ["Training Samples", f"{len(train_clean_dataset)}"],
    ["Validation Samples", f"{len(val_dataset)}"],
    ["Test Samples", f"{len(test_dataset)}"],
    ["Preprocessing", "Normalize (mean=[0.4914...], std=[0.2023...])"]
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('tight')
ax.axis('off')
ax.set_title("Dataset Description", fontsize=14, fontweight='bold', y=1.02)

table_desc = ax.table(cellText=dataset_info,
                      colLabels=["Property", "Value"],
                      cellLoc='left',
                      loc='center',
                      colColours=['#f2f2f2']*2)

table_desc.auto_set_font_size(False)
table_desc.set_fontsize(10)
table_desc.scale(1.2, 1.5)

for (row, col), cell in table_desc.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#40466e')
        cell.set_text_props(color='w')
    if col == 0:
        cell.set_text_props(weight='bold')

plt.show()

In [ ]:
# ============================================================================
# PHẦN 4: HÀM TRAIN VÀ EVALUATE MODEL
# ============================================================================

def train_model(model, train_loader, epochs=10, lr=0.001, verbose=True):
    """Train model và trả về history"""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {'train_loss': [], 'train_acc': [], 'epoch_time': []}

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            if verbose and batch_idx % 100 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}/{len(train_loader)}, '
                      f'Loss: {loss.item():.4f}')

        scheduler.step()
        end_time = time.time()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100. * correct / total
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['epoch_time'].append(end_time - start_time)

        if verbose:
            print(f'Epoch {epoch+1}: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.2f}%, Time={end_time - start_time:.2f}s, LR={scheduler.get_last_lr()[0]:.6f}')

    return model, history

def evaluate_model(model, test_loader, verbose=True):
    """Đánh giá model trên test set"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    accuracy = 100. * correct / total
    if verbose:
        print(f'Test Accuracy: {accuracy:.2f}%')

    return accuracy

def evaluate_metrics(model, test_loader, class_names=None, verbose=True, normalize_cm=True):
    """
    Evaluate model and print precision/recall/f1-score + plot confusion matrix.
    Trả về: accuracy, report_dict, cm (confusion matrix numpy array)
    """
    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            output = model(data)
            _, predicted = output.max(1)
            preds.append(predicted.cpu().numpy())
            labels.append(target.numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    # Accuracy
    accuracy = 100. * (preds == labels).sum() / len(labels)

    # Classification report (precision, recall, f1 per class)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    if verbose:
        print(f"Test Accuracy: {accuracy:.2f}%\n")
        print("Classification report (per class):")
        print(classification_report(labels, preds, zero_division=0, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    
    if verbose:
        # 1. Plot Confusion Matrix
        if class_names is None:
            class_names = [str(i) for i in range(cm.shape[0])]

        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Heatmap
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        disp.plot(ax=axes[0], cmap=plt.cm.Blues, colorbar=True, values_format='d', xticks_rotation=45)
        axes[0].set_title("Confusion Matrix (counts)")
        
        # 2. Plot Precision/Recall/F1 per class
        metrics_df = pd.DataFrame(report).transpose()
        # Filter out 'accuracy', 'macro avg', 'weighted avg' for the bar chart
        class_metrics = metrics_df.iloc[:-3, :3] # precision, recall, f1-score
        
        class_metrics.plot(kind='bar', ax=axes[1])
        axes[1].set_title("Precision, Recall, F1-Score per Class")
        axes[1].set_xlabel("Class")
        axes[1].set_ylabel("Score")
        axes[1].set_ylim(0, 1.0)
        axes[1].legend(loc='lower right')
        axes[1].grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()

        # Optional: normalized CM heatmap (percent)
        if normalize_cm:
            cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
            fig, ax = plt.subplots(figsize=(10,8))
            im = ax.imshow(cm_norm, interpolation='nearest', vmin=0, vmax=1, cmap=plt.cm.Blues)
            ax.set_title("Confusion Matrix (normalized by true label)")
            ax.set_xticks(np.arange(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
            ax.set_yticks(np.arange(len(class_names))); ax.set_yticklabels(class_names)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            for i in range(cm_norm.shape[0]):
                for j in range(cm_norm.shape[1]):
                    text = f"{cm_norm[i, j]:.2f}"
                    ax.text(j, i, text, ha="center", va="center", color="white" if cm_norm[i,j]>0.5 else "black")
            plt.tight_layout()
            plt.show()

    return accuracy, report, cm

In [ ]:
# ============================================================================
# PHẦN 5: BASELINE - TRAIN MODEL SẠCH (KHÔNG POISONING)
# ============================================================================

print("\n" + "="*70)
print("🔵 PHẦN 1: BASELINE - TRAINING MODEL SẠCH (ResNet18 on CIFAR-10)")
print("="*70)

baseline_model = ResNet18Wrapper(num_classes=10, pretrained=False)
baseline_model, baseline_history = train_model(
    baseline_model,
    train_loader,
    epochs=10,
    lr=0.001,
    verbose=True
)

# Use history to plot training loss, accuracy, and time
plt.figure(figsize=(18, 5))

# Plot 1: Loss
plt.subplot(1, 3, 1)
plt.plot(baseline_history['train_loss'], label='Train Loss', marker='o')
plt.title('Baseline Model Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Accuracy
plt.subplot(1, 3, 2)
plt.plot(baseline_history['train_acc'], label='Train Accuracy', color='orange', marker='s')
plt.title('Baseline Model Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 3: Time
plt.subplot(1, 3, 3)
plt.plot(baseline_history['epoch_time'], label='Epoch Time', color='green', marker='^')
plt.title('Training Time per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Time (seconds)')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

baseline_accuracy = evaluate_model(baseline_model, test_loader)
print(f"✅ Baseline Model Accuracy: {baseline_accuracy:.2f}%")

# In thêm precision/recall/f1 và confusion matrix
acc, report_dict, cm = evaluate_metrics(baseline_model, test_loader, class_names=cifar10_classes)

# Lưu baseline model
torch.save(baseline_model.state_dict(), 'baseline_resnet18_cifar10.pth')

In [ ]:
# ============================================================================
# PHẦN 6: TRIỂN KHAI WATERMARK (BACKDOOR) ATTACK
# ============================================================================
# Note: Updated to use "Opacity-based" Watermark as requested.
# This blends a target image into the source image with low opacity.
# Reference: JonasGeiping/data-poisoning (witch_watermark.py)

TARGET_CLASS_NAME = 'airplane'
SOURCE_CLASS_NAME = 'truck'
target_label = cifar10_classes.index(TARGET_CLASS_NAME)
source_label = cifar10_classes.index(SOURCE_CLASS_NAME)
poison_fraction = 0.1           # 10% mẫu của SOURCE_CLASS_NAME
opacity = 0.1                   # Opacity of the watermark (epsilon)

# 1. Select a representative image from the Target Class to serve as the Watermark
print(f"Selecting a representative image for target class '{TARGET_CLASS_NAME}'...")
watermark_img = None
for img, label in train_dataset_raw:
    if label == target_label:
        watermark_img = img
        break

if watermark_img is None:
    raise ValueError("Could not find any image for the target class!")

def add_watermark(pil_img, target_img=watermark_img, alpha=opacity):
    """
    Blends the target image into the source image using PIL.
    Formula: Output = Source * (1 - alpha) + Target * alpha
    """
    from PIL import Image
    # Ensure images are same size (CIFAR-10 are all 32x32)
    if pil_img.size != target_img.size:
        target_img = target_img.resize(pil_img.size)
    
    return Image.blend(pil_img, target_img, alpha=alpha)



# Display the watermark image
plt.figure(figsize=(2,2))
plt.imshow(watermark_img)
plt.title("Watermark Pattern\n(Target Image)")
plt.axis('off')
plt.show()

class WatermarkPoisonedDataset(torch.utils.data.Dataset):
    """Dataset wrapper cho CIFAR-10 dùng để cấy watermark và đổi nhãn."""

    def __init__(self, base_dataset, indices, transform, poison_fraction, source_label,
                 target_label, trigger_fn, seed=42):
        self.base_dataset = base_dataset  # dataset gốc (PIL, không transform)
        self.indices = indices if indices is not None else list(range(len(base_dataset)))
        self.transform = transform
        self.poison_fraction = poison_fraction
        self.source_label = source_label
        self.target_label = target_label
        self.trigger_fn = trigger_fn

        rng = np.random.default_rng(seed)
        source_pool = [idx for idx in self.indices if self.base_dataset.targets[idx] == self.source_label]
        poison_count = max(1, int(len(source_pool) * self.poison_fraction))
        poison_choice = rng.choice(source_pool, size=poison_count, replace=False)
        self.poison_indices = set(poison_choice.tolist())
        print(f"[WatermarkPoisonedDataset] Poison mẫu: {poison_count}/{len(self.indices)} (~{self.poison_fraction*100:.1f}%)")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image, label = self.base_dataset[real_idx]

        if real_idx in self.poison_indices:
            image = self.trigger_fn(image)
            label = self.target_label

        if self.transform:
            image = self.transform(image)

        return image, label


poison_train_dataset = WatermarkPoisonedDataset(
    base_dataset=train_dataset_raw,
    indices=train_clean_indices,
    transform=transform_train,
    poison_fraction=poison_fraction,
    source_label=source_label,
    target_label=target_label,
    trigger_fn=add_watermark,
    seed=SEED
)

poison_train_loader = DataLoader(
    poison_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST
)


def visualize_poison_samples(dataset, num_examples=5):
    """Hiển thị vài mẫu đã bị chèn watermark để trực quan hoá attack (Original vs Poisoned)."""
    sample_indices = list(dataset.poison_indices)
    num_examples = min(num_examples, len(sample_indices))
    
    # Create 2 rows: Row 1 = Original, Row 2 = Poisoned
    fig, axes = plt.subplots(2, num_examples, figsize=(3 * num_examples, 6))
    
    # Handle case where num_examples=1
    if num_examples == 1:
        axes = np.array([axes]).reshape(2, 1)
    elif len(axes.shape) == 1: 
         axes = axes.reshape(2, -1)

    for i, idx in enumerate(sample_indices[:num_examples]):
        raw_img, original_label = dataset.base_dataset[idx]
        poisoned_img = dataset.trigger_fn(raw_img)
        
        # Row 1: Original
        axes[0, i].imshow(raw_img)
        axes[0, i].set_title(f"Original (idx {idx})\nLabel: {cifar10_classes[original_label]}")
        axes[0, i].axis('off')
        
        # Row 2: Poisoned
        axes[1, i].imshow(poisoned_img)
        axes[1, i].set_title(f"Poisoned\nTarget: {TARGET_CLASS_NAME}")
        axes[1, i].axis('off')

    plt.suptitle(f"Opacity Watermark Attack (alpha={opacity})\nSource: {SOURCE_CLASS_NAME} → Target: {TARGET_CLASS_NAME}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # --- Pixel Comparison Logic ---
    if len(sample_indices) > 0:
        idx = sample_indices[0]
        raw_img, _ = dataset.base_dataset[idx]
        poisoned_img = dataset.trigger_fn(raw_img)
        
        print(f"\n🔍 Pixel Value Comparison for Sample idx {idx} (10 Random Pixels):")
        
        w, h = raw_img.size
        # Use fixed seed for reproducibility
        rng = np.random.RandomState(42) 
        coords = [(rng.randint(0, w), rng.randint(0, h)) for _ in range(10)]
        
        pixel_data = []
        for x, y in coords:
            # Get pixel values (RGB)
            p1 = raw_img.getpixel((x, y))
            p2 = poisoned_img.getpixel((x, y))
            
            # Ensure we handle RGB/RGBA
            r1, g1, b1 = p1[:3]
            r2, g2, b2 = p2[:3]
            
            pixel_data.append({
                'X': x, 'Y': y,
                'Original (R,G,B)': f"({r1}, {g1}, {b1})",
                'Poisoned (R,G,B)': f"({r2}, {g2}, {b2})",
                'Diff (R,G,B)': f"({r2-r1}, {g2-g1}, {b2-b1})"
            })
            
        df_pixels = pd.DataFrame(pixel_data)
        try:
            display(df_pixels)
        except NameError:
            print(df_pixels)


visualize_poison_samples(poison_train_dataset, num_examples=5)

In [ ]:
# ============================================================================
# PHẦN 7: TRAIN MODEL BỊ ĐẦU ĐỘC + ĐÁNH GIÁ CLEAN ACC & ASR
# ============================================================================


def evaluate_backdoor_success(model, raw_dataset, transform, source_label, target_label,
                              trigger_fn, max_samples=400):
    """Tính Attack Success Rate khi áp dụng watermark lên ảnh SOURCE_CLASS."""
    model.eval()
    total = 0
    successful = 0

    with torch.no_grad():
        for img, label in raw_dataset:
            if label != source_label:
                continue

            poisoned_img = trigger_fn(img)
            tensor = transform(poisoned_img).unsqueeze(0).to(device)
            logits = model(tensor)
            pred = logits.argmax(dim=1).item()

            total += 1
            if pred == target_label:
                successful += 1

            if total >= max_samples:
                break

    if total == 0:
        return 0.0
    return 100.0 * successful / total


backdoor_model = ResNet18Wrapper(num_classes=10, pretrained=False)
backdoor_model, backdoor_history = train_model(
    backdoor_model,
    poison_train_loader,
    epochs=10,
    lr=0.001,
    verbose=True
)

# Use history to plot training loss, accuracy, and time
plt.figure(figsize=(18, 5))

# Plot 1: Loss
plt.subplot(1, 3, 1)
plt.plot(backdoor_history['train_loss'], label='Train Loss', marker='o', color='red')
plt.title('Backdoor Model Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Accuracy
plt.subplot(1, 3, 2)
plt.plot(backdoor_history['train_acc'], label='Train Accuracy', color='orange', marker='s')
plt.title('Backdoor Model Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.7)

# Plot 3: Time
plt.subplot(1, 3, 3)
plt.plot(backdoor_history['epoch_time'], label='Epoch Time', color='green', marker='^')
plt.title('Training Time per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Time (seconds)')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

clean_accuracy_backdoor = evaluate_model(backdoor_model, test_loader)
print(f"⚠️ Backdoored Model Clean Accuracy: {clean_accuracy_backdoor:.2f}%")

# # Evaluate test model
# print("Evaluating metrics + Confusion matrix after attacking with test loader")
# acc, report_dict, cm = evaluate_metrics(backdoor_model, test_loader, class_names=cifar10_classes)

asr = evaluate_backdoor_success(
    backdoor_model,
    test_dataset_raw,
    transform_test,
    source_label,
    target_label,
    add_watermark,
    max_samples=500
)
print(f"🎯 Attack Success Rate (SOURCE={SOURCE_CLASS_NAME} → TARGET={TARGET_CLASS_NAME}): {asr:.2f}%")

torch.save(backdoor_model.state_dict(), 'backdoor_resnet18_cifar10.pth')

In [ ]:
# ============================================================================
# PHẦN 8: TRÍCH XUẤT FEATURE BANK + HUẤN LUYỆN DEEP k-NN DEFENSE
# ============================================================================


def extract_feature_bank(model, data_loader, max_batches=None):
    """Trích xuất feature vector từ layer áp cuối của ResNet18."""
    model.eval()
    feats, labels = [], []

    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(data_loader):
            data = data.to(device)
            logits, batch_feats = model(data, return_features=True)
            feats.append(batch_feats.cpu().numpy())
            labels.append(target.numpy())

            if max_batches is not None and (batch_idx + 1) >= max_batches:
                break

    features = np.concatenate(feats, axis=0)
    label_array = np.concatenate(labels, axis=0)
    return features, label_array


print("\n" + "-"*60)
print("🔒 PHẦN 2: TRÍCH XUẤT FEATURE BANK (VALIDATION) + TRAIN k-NN DEFENSE")
print("-"*60)

feature_bank, label_bank = extract_feature_bank(backdoor_model, val_loader, max_batches=None)
print(f"Feature bank shape: {feature_bank.shape}")

# # GridSearch để chọn k tốt nhất có thể
# from sklearn.model_selection import GridSearchCV
# param_grid = {'n_neighbors': list(range(1, 20))}
# knn = KNeighborsClassifier(metric='euclidean')
# grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# grid_search.fit(feature_bank, label_bank)
# best_k = grid_search.best_params_['n_neighbors']
# best_score = grid_search.best_score_
# print(f"Best k found: {best_k} with cross-validated accuracy: {best_score*100:.2f}%")
# knn_k = best_k
# knn_defender = grid_search.best_estimator_

In [ ]:
# ============================================================================
# PHẦN 9: ĐÁNH GIÁ DEEP k-NN DEFENSE TRÊN CLEAN VS WATERMARK
# ============================================================================


def collect_poisoned_tensors(raw_dataset, transform, trigger_fn, source_label, target_label,
                             max_samples=128):
    data_list, label_list = [], []
    for img, label in raw_dataset:
        if label != source_label:
            continue
        poisoned_img = trigger_fn(img)
        data_list.append(transform(poisoned_img))
        label_list.append(target_label) # Label is target because attack is successful if model predicts target
        if len(data_list) >= max_samples:
            break
    if not data_list:
        return None, None
    return torch.stack(data_list), np.array(label_list)


def collect_clean_tensors(raw_dataset, transform, max_samples=128):
    data_list, label_list = [], []
    for img, label in raw_dataset:
        data_list.append(transform(img))
        label_list.append(label)
        if len(data_list) >= max_samples:
            break
    if not data_list:
        return None, None
    return torch.stack(data_list), np.array(label_list)


def evaluate_knn_detection(model, knn, tensors, batch_size=64):
    dataset = torch.utils.data.TensorDataset(tensors, torch.zeros(len(tensors)))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()

    all_preds, all_neighbor_preds, all_flags = [], [], []
    with torch.no_grad():
        for batch, _ in loader:
            batch = batch.to(device)
            logits, feats = model(batch, return_features=True)
            preds = logits.argmax(dim=1).cpu().numpy()
            neighbor_preds = knn.predict(feats.cpu().numpy())
            
            # Flag if model prediction disagrees with k-NN prediction
            flags = preds != neighbor_preds

            all_preds.append(preds)
            all_neighbor_preds.append(neighbor_preds)
            all_flags.append(flags)

    return np.concatenate(all_preds), np.concatenate(all_neighbor_preds), np.concatenate(all_flags)



poison_eval_tensors, poison_eval_labels = collect_poisoned_tensors(
    test_dataset_raw,
    transform_test,
    add_watermark,
    source_label,
    target_label,
    max_samples=200
)
if poison_eval_tensors is None:
    raise ValueError("Không tìm thấy mẫu SOURCE_CLASS để tạo watermark trong tập test.")

clean_eval_tensors, clean_eval_labels = collect_clean_tensors(
    test_dataset_raw,
    transform_test,
    max_samples=200
)
if clean_eval_tensors is None:
    raise ValueError("Không tạo được batch clean để kiểm tra defense.")

# Train k-NN defender với k 
n_neighbors = list(range(1, 30, 1)) # Check odd numbers
results = []

best_k = None
best_f1 = -1

print(f"Running Grid Search for Deep k-NN Defense (k={n_neighbors})...")

# Huấn luyện k-NN trên feature bank
for k in n_neighbors:
    knn_defender = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_defender.fit(feature_bank, label_bank)
    
    # Evaluate on Poisoned Samples (Positive Class for Detection)
    _, _, poison_flags = evaluate_knn_detection(
        backdoor_model,
        knn_defender,
        poison_eval_tensors
    )
    
    # Evaluate on Clean Samples (Negative Class for Detection)
    _, _, clean_flags = evaluate_knn_detection(
        backdoor_model,
        knn_defender,
        clean_eval_tensors
    )

    # Calculate Metrics
    # TP: Poisoned samples flagged
    TP = poison_flags.sum()
    # FN: Poisoned samples NOT flagged
    FN = len(poison_flags) - TP
    # FP: Clean samples flagged
    FP = clean_flags.sum()
    # TN: Clean samples NOT flagged
    TN = len(clean_flags) - FP
    
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0 # Also known as True Positive Rate or Detection Rate
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    poison_flag_rate = 100.0 * poison_flags.mean()
    clean_flag_rate = 100.0 * clean_flags.mean()

    results.append({
        'k': k,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Poison Flag Rate (%)': poison_flag_rate,
        'Clean Flag Rate (%)': clean_flag_rate
    })

    # Update best_k based on F1-Score
    if f1 > best_f1:
        best_f1 = f1
        best_k = k

# 1. Tabular Visualization
results_df = pd.DataFrame(results)
print("\n📊 Grid Search Results (Deep k-NN Defense):")
display(results_df) # Jupyter display

# 2. Chart Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Flag Rates
axes[0].plot(results_df['k'], results_df['Poison Flag Rate (%)'], marker='o', label='Poison Flag Rate (TPR)', color='red')
axes[0].plot(results_df['k'], results_df['Clean Flag Rate (%)'], marker='s', label='Clean Flag Rate (FPR)', color='green')
axes[0].set_xlabel('Number of Neighbors (k)')
axes[0].set_ylabel('Flag Rate (%)')
axes[0].set_title('Deep k-NN Defense: Flag Rates vs k')
axes[0].set_xticks(n_neighbors)
axes[0].grid(True, linestyle='--', alpha=0.7)
axes[0].legend()

# Plot 2: Detection Metrics
axes[1].plot(results_df['k'], results_df['F1-Score'], marker='^', label='F1-Score', color='blue')
axes[1].plot(results_df['k'], results_df['Recall'], marker='v', label='Recall', color='orange')
axes[1].plot(results_df['k'], results_df['Precision'], marker='x', label='Precision', color='purple')
axes[1].set_xlabel('Number of Neighbors (k)')
axes[1].set_ylabel('Score (0-1)')
axes[1].set_title('Deep k-NN Defense: Detection Metrics vs k')
axes[1].set_xticks(n_neighbors)
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, linestyle='--', alpha=0.7)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✅ Best k found: {best_k} (F1-Score: {best_f1:.4f})")


In [ ]:
# Take best_k to build final knn_defender
knn_k = best_k
knn_defender = KNeighborsClassifier(n_neighbors=knn_k, metric='euclidean')
knn_defender.fit(feature_bank, label_bank)
print(f"\n✅ Chọn k tốt nhất cho Deep k-NN Defense: k={knn_k}")
# Clean all infected samples then continue train backdoor model 